# Meta Feature Analysis - Part 1

# Data Loading

In [44]:
from __future__ import annotations

import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

In [45]:
CONFIG = "config_1.yaml"

In [46]:
from dataclasses import replace

from mfa import load_config

config_path = project_dir / "configs" / CONFIG
config = load_config(config_path)

# Read the configured value from YAML
yaml_n_jobs = config.parallelism.n_jobs

# Keep this notebook safe: allow only 1..2 workers for this run
run_n_jobs = min(max(yaml_n_jobs, 1), 2)
effective_n_jobs = 2

# Build a temporary config override for this run only (does not edit YAML)
run_config = replace(
    config,
    parallelism=replace(config.parallelism, n_jobs=effective_n_jobs),
)

if yaml_n_jobs != run_n_jobs:
    print(
        f"Loaded {config_path.name}; parallelism.n_jobs={yaml_n_jobs}. "
        f"Temporarily using n_jobs={run_n_jobs} for this run."
    )
else:
    print(
        f"Loaded {config_path.name}; parallelism.n_jobs={yaml_n_jobs}. "
        "Proceeding with the analysis."
    )

Loaded config_1.yaml; parallelism.n_jobs=-1. Temporarily using n_jobs=1 for this run.


In [47]:
from mfa import run_analysis
from tabarena.nips2025_utils.fetch_metadata import load_task_metadata

result = run_analysis(run_config)

task_type_lookup = (
    load_task_metadata()[["dataset", "problem_type"]]
    .rename(columns={"problem_type": "task_type"})
    .drop_duplicates(subset="dataset")
)
result.metafeature_table = result.metafeature_table.merge(
    task_type_lookup, on="dataset", how="left", validate="many_to_one"
)
result.analysis_table = result.analysis_table.merge(
    task_type_lookup, on="dataset", how="left", validate="many_to_one"
)
result.analysis_table.head()

13:59:17 INFO mfa.pipeline: Starting analysis: comparisons=non_foundational_vs_foundational; scope=all benchmark datasets; unit=dataset; method_variant=default,tuned; n_jobs=2
13:59:17 INFO mfa.pipeline: Stage 1/5 raw results: cache hit (34110 rows, 51 dataset(s))
13:59:17 INFO mfa.pipeline: Stage 2/5 meta-features: trace enabled; metafeature caches remain active, so live per-split diagnostics appear only on cache misses
13:59:17 INFO mfa.pipeline: Stage 2/5 meta-features: pymfe enabled; rebuilding from split cache and reusing cached pymfe failures/incomplete outputs as-is
13:59:17 INFO mfa.pipeline: Stage 2/5 meta-features: building for all benchmark datasets
13:59:17 INFO mfa.metafeatures: Meta-features: preparing 51 dataset(s) with feature_sets=basic,redundancy,pymfe (n_jobs=2)
13:59:17 INFO mfa.metafeatures: Meta-features: trace enabled; split cache remains active, so live timing and warning diagnostics appear only on cache misses
13:59:17 INFO mfa.metafeatures: Meta-features: trac

,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,...,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem,task_type
0,APSFailure,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,50666.666667,170.0,...,0.832294,0.007284,0.333333,0.001366,0.002011,0.000670,0.498960,0.549148,0.183049,binary
1,Amazon_employee_access,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,21846.000000,9.0,...,0.001416,0.146592,0.778336,-0.029811,0.007122,0.002374,-0.776919,0.115570,0.038523,binary
2,Another-Dataset-on-used-Fiat-500,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,30,1025.333333,7.0,...,0.502696,719.852165,0.203675,12.029602,16.598990,3.030547,0.299021,0.414145,0.075612,regression
3,Bank_Customer_Churn,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,6666.666667,10.0,...,0.479097,0.125676,0.037007,0.004284,0.001790,0.000597,0.442090,0.159837,0.053279,binary
4,Bioresponse,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,2500.666667,1776.0,...,0.441583,0.123178,0.216194,0.003073,0.005005,0.001668,0.225389,0.360046,0.120015,binary


# Preprocessing

In [48]:
import pandas as pd

# -- Inspect what the result object contains --
print(f"config_hash:        {result.config_hash}")
print(f"comparison_name:    {result.comparison_name}")
print(f"analysis_table:     {result.analysis_table.shape}")
print(f"gap_table:          {result.gap_table.shape}")
print(f"metafeature_table:  {result.metafeature_table.shape}")
print(f"correlation_results: {len(result.correlation_results)} features tested")
print(
    f"correction_result:  {result.correction_result.method if result.correction_result else None}"
)
print(f"multivariate_result: {result.multivariate_result}")

config_hash:        c4472762a9293af4
comparison_name:    non_foundational_vs_foundational
analysis_table:     (51, 1133)
gap_table:          (816, 17)
metafeature_table:  (816, 1118)
correlation_results: 1114 features tested
correction_result:  bh
multivariate_result: None


## Build Task-Aware Analysis Tables

We construct two analysis matrices because the available meta-features depend on the supervised task type. The general matrix contains only features that are meaningful for both regression and classification tasks. The classification matrix is restricted to classification tasks and augments the general feature set with class-label-dependent features such as class counts, entropy, imbalance, and Pymfe classification-only descriptors.

The two matrices are analyzed separately downstream, so their preprocessing filters are estimated within their own task populations rather than forcing the classification-specific analysis to inherit decisions from the full regression-plus-classification table.

In [49]:
import numpy as np

from mfa.metafeatures.basic import (
    CLASSIFICATION_PROBLEM_TYPES as BASIC_CLASSIFICATION_PROBLEM_TYPES,
)
from mfa.metafeatures.pymfe_catalog import PYMFE_CLASSIFICATION_ONLY
from mfa.stats.correlation import EXCLUDED_PREDICTOR_COLUMNS

BASIC_CLASSIFICATION_ONLY_FEATURES = frozenset(
    {
        "n_classes",
        "class_entropy",
        "majority_class_fraction",
        "minority_class_fraction",
        "class_imbalance_ratio",
    }
)
CLASSIFICATION_PROBLEM_TYPES = set(BASIC_CLASSIFICATION_PROBLEM_TYPES)

ANALYSIS_CONTEXT_COLUMNS = [
    "dataset",
    "task_type",
    "comparison_name",
    "group_a_name",
    "group_b_name",
    "group_a_label",
    "group_b_label",
    "expected_direction",
    "n_splits",
    "best_a_error",
    "best_a_norm_error",
    "best_b_error",
    "best_b_norm_error",
    "delta_raw",
    "delta_raw_std",
    "delta_raw_sem",
    "delta_norm",
    "delta_norm_std",
    "delta_norm_sem",
]


def _is_pymfe_classification_only(column: str) -> bool:
    if not column.startswith("pymfe__"):
        return False
    raw_feature = column.removeprefix("pymfe__").split(".", maxsplit=1)[0]
    return raw_feature in PYMFE_CLASSIFICATION_ONLY


def is_classification_only_feature(column: str) -> bool:
    return (
        column in BASIC_CLASSIFICATION_ONLY_FEATURES
        or _is_pymfe_classification_only(column)
    )


def infer_numeric_predictors(
    table: pd.DataFrame, *, target: str = "delta_norm"
) -> list[str]:
    predictors = []
    for column in table.columns:
        if (
            column == target
            or column in EXCLUDED_PREDICTOR_COLUMNS
            or column.startswith("best_")
        ):
            continue
        numeric_values = pd.to_numeric(table[column], errors="coerce")
        if pd.api.types.is_numeric_dtype(table[column]) or numeric_values.notna().any():
            predictors.append(column)
    return predictors


available_context_columns = [
    column
    for column in ANALYSIS_CONTEXT_COLUMNS
    if column in result.analysis_table.columns
]
all_predictor_columns = infer_numeric_predictors(result.analysis_table)
shared_predictor_columns = [
    column
    for column in all_predictor_columns
    if not is_classification_only_feature(column)
]
classification_only_predictor_columns = [
    column for column in all_predictor_columns if is_classification_only_feature(column)
]
classification_predictor_columns = [
    *shared_predictor_columns,
    *classification_only_predictor_columns,
]

task_type = result.analysis_table["task_type"].astype("string").str.lower()
classification_mask = task_type.isin(CLASSIFICATION_PROBLEM_TYPES)

analysis_general_raw = result.analysis_table.loc[
    :, available_context_columns + shared_predictor_columns
].copy()
analysis_classification_raw = result.analysis_table.loc[
    classification_mask, available_context_columns + classification_predictor_columns
].copy()

feature_table_summary = pd.DataFrame(
    [
        {
            "table": "general_regression_and_classification",
            "rows": len(analysis_general_raw),
            "predictor_columns": len(shared_predictor_columns),
            "classification_only_predictors": 0,
        },
        {
            "table": "classification_augmented",
            "rows": len(analysis_classification_raw),
            "predictor_columns": len(classification_predictor_columns),
            "classification_only_predictors": len(
                classification_only_predictor_columns
            ),
        },
    ]
)

display(feature_table_summary)
print(
    f"Shared predictors retained before preprocessing: {len(shared_predictor_columns)}"
)
print(
    f"Classification-only predictors available before preprocessing: {len(classification_only_predictor_columns)}"
)

,table,rows,predictor_columns,classification_only_predictors
0,general_regression_and_classification,51,544,0
1,classification_augmented,38,1114,570


Shared predictors retained before preprocessing: 544
Classification-only predictors available before preprocessing: 570


## Preprocess Analysis Tables

For each analysis matrix, infinite values are treated as missing values before feature filtering. We then remove features with high missingness and features with negligible empirical variation. The near-constant filter removes exact constants, numerically constant features under a tight relative tolerance, and features where one rounded value dominates at least 95% of non-missing observations. These rules are unsupervised and are applied independently to the general and classification-specific matrices.

In [50]:
MAX_FEATURE_MISSINGNESS = 0.20
NEAR_CONSTANT_TOP_SHARE = 0.95
NUMERICAL_CONSTANT_REL_TOL = 1e-12


def _near_constant_report(
    table: pd.DataFrame,
    feature_columns: list[str],
    *,
    top_share_threshold: float = NEAR_CONSTANT_TOP_SHARE,
    numerical_constant_rel_tol: float = NUMERICAL_CONSTANT_REL_TOL,
) -> pd.DataFrame:
    rows = []
    for column in feature_columns:
        values = pd.to_numeric(table[column], errors="coerce").dropna()
        n_non_missing = int(values.shape[0])
        if n_non_missing == 0:
            rows.append(
                {
                    "feature": column,
                    "n_non_missing": n_non_missing,
                    "n_unique": 0,
                    "top_share": np.nan,
                    "reason": "all_missing_after_inf_replacement",
                }
            )
            continue

        n_unique = int(values.nunique(dropna=True))
        rounded_values = values.round(12)
        top_share = float(
            rounded_values.value_counts(normalize=True, dropna=True).iloc[0]
        )
        scale = max(1.0, float(values.abs().median()))
        numerical_span = float(values.max() - values.min())

        reason = None
        if n_unique <= 1:
            reason = "exact_constant"
        elif numerical_span <= numerical_constant_rel_tol * scale:
            reason = "numerical_constant"
        elif top_share >= top_share_threshold:
            reason = "dominant_value_share_ge_0.95"

        if reason is not None:
            rows.append(
                {
                    "feature": column,
                    "n_non_missing": n_non_missing,
                    "n_unique": n_unique,
                    "top_share": top_share,
                    "reason": reason,
                }
            )
    return pd.DataFrame(rows)


def preprocess_analysis_table(
    table: pd.DataFrame,
    feature_columns: list[str],
    *,
    table_name: str,
    max_feature_missingness: float = MAX_FEATURE_MISSINGNESS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cleaned = table.copy()
    for column in cleaned.columns:
        numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
        if (
            pd.api.types.is_numeric_dtype(cleaned[column])
            or numeric_values.notna().any()
        ):
            cleaned[column] = numeric_values.replace([np.inf, -np.inf], np.nan)
    feature_columns = [
        column for column in feature_columns if column in cleaned.columns
    ]

    missing_fraction = (
        cleaned[feature_columns].isna().mean().sort_values(ascending=False)
    )
    high_missing_features = missing_fraction[
        missing_fraction > max_feature_missingness
    ].index.tolist()
    after_missingness = [
        column for column in feature_columns if column not in high_missing_features
    ]

    near_constant = _near_constant_report(cleaned, after_missingness)
    near_constant_features = (
        near_constant["feature"].tolist() if not near_constant.empty else []
    )
    retained_features = [
        column for column in after_missingness if column not in near_constant_features
    ]

    context_columns = [
        column for column in available_context_columns if column in cleaned.columns
    ]
    processed = cleaned.loc[:, context_columns + retained_features].copy()

    report_rows = []
    for feature in high_missing_features:
        report_rows.append(
            {
                "table": table_name,
                "stage": "high_missingness",
                "feature": feature,
                "missing_fraction": float(missing_fraction.loc[feature]),
                "reason": f"> {max_feature_missingness:.0%} missing",
            }
        )
    if not near_constant.empty:
        near_constant = near_constant.assign(table=table_name, stage="near_constant")
        near_constant["missing_fraction"] = (
            near_constant["feature"].map(missing_fraction).astype(float)
        )
        report_rows.extend(
            near_constant[
                [
                    "table",
                    "stage",
                    "feature",
                    "missing_fraction",
                    "n_non_missing",
                    "n_unique",
                    "top_share",
                    "reason",
                ]
            ].to_dict("records")
        )

    report = pd.DataFrame(report_rows)
    print(
        f"{table_name}: rows={len(processed)}, predictors={len(retained_features)} "
        f"(dropped {len(high_missing_features)} high-missing, {len(near_constant_features)} near-constant)"
    )
    return processed, report


analysis_general, general_preprocessing_report = preprocess_analysis_table(
    analysis_general_raw,
    shared_predictor_columns,
    table_name="general_regression_and_classification",
)
analysis_classification, classification_preprocessing_report = (
    preprocess_analysis_table(
        analysis_classification_raw,
        classification_predictor_columns,
        table_name="classification_augmented",
    )
)

preprocessing_report = pd.concat(
    [general_preprocessing_report, classification_preprocessing_report],
    ignore_index=True,
)
preprocessing_summary = pd.DataFrame(
    [
        {
            "table": "general_regression_and_classification",
            "rows": len(analysis_general),
            "columns": analysis_general.shape[1],
            "predictors_after_preprocessing": analysis_general.shape[1]
            - len(available_context_columns),
        },
        {
            "table": "classification_augmented",
            "rows": len(analysis_classification),
            "columns": analysis_classification.shape[1],
            "predictors_after_preprocessing": analysis_classification.shape[1]
            - len(available_context_columns),
        },
    ]
)

display(preprocessing_summary)
display(preprocessing_report)

general_regression_and_classification: rows=51, predictors=488 (dropped 56 high-missing, 0 near-constant)
classification_augmented: rows=38, predictors=1022 (dropped 62 high-missing, 30 near-constant)


,table,rows,columns,predictors_after_preprocessing
0,general_regression_and_classification,51,507,488
1,classification_augmented,38,1041,1022


,table,stage,feature,missing_fraction,reason,n_non_missing,n_unique,top_share
0,general_regression_and_classification,high_missingness,pymfe__h_mean.skewness,0.470588,> 20% missing,NaN,NaN,NaN
1,general_regression_and_classification,high_missingness,pymfe__h_mean.kurtosis,0.470588,> 20% missing,NaN,NaN,NaN
2,general_regression_and_classification,high_missingness,pymfe__g_mean.kurtosis,0.470588,> 20% missing,NaN,NaN,NaN
3,general_regression_and_classification,high_missingness,pymfe__g_mean.skewness,0.470588,> 20% missing,NaN,NaN,NaN
4,general_regression_and_classification,high_missingness,pymfe__g_mean.histogram.6,0.352941,> 20% missing,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
143,classification_augmented,near_constant,pymfe__linear_discr.count,0.105263,dominant_value_share_ge_0.95,34.0,2.0,0.970588
144,classification_augmented,near_constant,pymfe__naive_bayes.count,0.105263,exact_constant,34.0,1.0,1.000000
145,classification_augmented,near_constant,pymfe__one_nn.count,0.105263,exact_constant,34.0,1.0,1.000000
146,classification_augmented,near_constant,pymfe__random_node.count,0.105263,exact_constant,34.0,1.0,1.000000


## Reduce Redundant Features

We reduce redundancy using only feature-feature relationships, not the downstream performance target. Within each analysis matrix, we compute absolute Spearman correlations among the preprocessed predictors and remove one feature from pairs with near-duplicate rank behavior. For each highly correlated pair, the retained feature is chosen deterministically by lower missingness, then more unique observed values, then column name. This keeps the filter unsupervised, reproducible, scale-invariant, and separately calibrated to the general and classification-specific analysis populations.

In [51]:
REDUNDANCY_SPEARMAN_THRESHOLD = 0.95


def _redundancy_feature_columns(table: pd.DataFrame) -> list[str]:
    return [
        column
        for column in table.columns
        if column not in available_context_columns and column != "delta_norm"
    ]


def _prefer_feature(
    feature_a: str,
    feature_b: str,
    *,
    missing_fraction: pd.Series,
    n_unique: pd.Series,
) -> tuple[str, str]:
    key_a = (
        float(missing_fraction.loc[feature_a]),
        -int(n_unique.loc[feature_a]),
        feature_a,
    )
    key_b = (
        float(missing_fraction.loc[feature_b]),
        -int(n_unique.loc[feature_b]),
        feature_b,
    )
    if key_a <= key_b:
        return feature_a, feature_b
    return feature_b, feature_a


def reduce_redundant_features(
    table: pd.DataFrame,
    *,
    table_name: str,
    threshold: float = REDUNDANCY_SPEARMAN_THRESHOLD,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_columns = _redundancy_feature_columns(table)
    if len(feature_columns) <= 1:
        return table.copy(), pd.DataFrame()

    numeric_features = table[feature_columns].apply(pd.to_numeric, errors="coerce")
    missing_fraction = numeric_features.isna().mean()
    n_unique = numeric_features.nunique(dropna=True)
    corr = numeric_features.corr(method="spearman", min_periods=3).abs()

    high_corr_pairs = []
    for i, feature_a in enumerate(feature_columns):
        for feature_b in feature_columns[i + 1 :]:
            abs_spearman = corr.loc[feature_a, feature_b]
            if pd.notna(abs_spearman) and abs_spearman >= threshold:
                high_corr_pairs.append((float(abs_spearman), feature_a, feature_b))
    high_corr_pairs.sort(key=lambda row: (-row[0], row[1], row[2]))

    dropped_features: set[str] = set()
    report_rows = []
    for abs_spearman, feature_a, feature_b in high_corr_pairs:
        if feature_a in dropped_features or feature_b in dropped_features:
            continue

        kept_feature, dropped_feature = _prefer_feature(
            feature_a,
            feature_b,
            missing_fraction=missing_fraction,
            n_unique=n_unique,
        )
        dropped_features.add(dropped_feature)
        report_rows.append(
            {
                "table": table_name,
                "stage": "redundancy",
                "dropped_feature": dropped_feature,
                "kept_feature": kept_feature,
                "abs_spearman": abs_spearman,
                "dropped_missing_fraction": float(
                    missing_fraction.loc[dropped_feature]
                ),
                "kept_missing_fraction": float(missing_fraction.loc[kept_feature]),
                "dropped_n_unique": int(n_unique.loc[dropped_feature]),
                "kept_n_unique": int(n_unique.loc[kept_feature]),
                "reason": f"abs_spearman >= {threshold:.2f}",
            }
        )

    retained_features = [
        feature for feature in feature_columns if feature not in dropped_features
    ]
    context_columns = [
        column for column in available_context_columns if column in table
    ]
    reduced = table.loc[:, context_columns + retained_features].copy()
    report = pd.DataFrame(report_rows)
    print(
        f"{table_name}: rows={len(reduced)}, predictors={len(retained_features)} "
        f"(dropped {len(dropped_features)} redundant)"
    )
    return reduced, report


analysis_general_reduced, general_redundancy_report = reduce_redundant_features(
    analysis_general,
    table_name="general_regression_and_classification",
)
analysis_classification_reduced, classification_redundancy_report = (
    reduce_redundant_features(
        analysis_classification,
        table_name="classification_augmented",
    )
)

redundancy_report = pd.concat(
    [general_redundancy_report, classification_redundancy_report],
    ignore_index=True,
)
redundancy_summary = pd.DataFrame(
    [
        {
            "table": "general_regression_and_classification",
            "rows": len(analysis_general_reduced),
            "columns": analysis_general_reduced.shape[1],
            "predictors_after_redundancy_reduction": len(
                _redundancy_feature_columns(analysis_general_reduced)
            ),
            "predictors_dropped_as_redundant": len(general_redundancy_report),
        },
        {
            "table": "classification_augmented",
            "rows": len(analysis_classification_reduced),
            "columns": analysis_classification_reduced.shape[1],
            "predictors_after_redundancy_reduction": len(
                _redundancy_feature_columns(analysis_classification_reduced)
            ),
            "predictors_dropped_as_redundant": len(classification_redundancy_report),
        },
    ]
)

display(redundancy_summary)
display(redundancy_report)

general_regression_and_classification: rows=51, predictors=284 (dropped 204 redundant)
classification_augmented: rows=38, predictors=556 (dropped 466 redundant)


,table,rows,columns,predictors_after_redundancy_reduction,predictors_dropped_as_redundant
0,general_regression_and_classification,51,303,284,204
1,classification_augmented,38,575,556,466


,table,stage,dropped_feature,kept_feature,abs_spearman,dropped_missing_fraction,kept_missing_fraction,dropped_n_unique,kept_n_unique,reason
0,general_regression_and_classification,redundancy,pymfe__cat_to_num,cat_fraction,1.000000,0.098039,0.000000,27,28,abs_spearman >= 0.95
1,general_regression_and_classification,redundancy,pymfe__attr_ent.count,d,1.000000,0.019608,0.000000,34,35,abs_spearman >= 0.95
2,general_regression_and_classification,redundancy,pymfe__nr_attr,d,1.000000,0.019608,0.000000,34,35,abs_spearman >= 0.95
3,general_regression_and_classification,redundancy,pymfe__sparsity.count,d,1.000000,0.117647,0.000000,32,35,abs_spearman >= 0.95
4,general_regression_and_classification,redundancy,pymfe__attr_to_inst,d_over_n,1.000000,0.019608,0.000000,50,51,abs_spearman >= 0.95
...,...,...,...,...,...,...,...,...,...,...
665,classification_augmented,redundancy,pymfe__leaves_corrob.iq_range,log_n,0.954062,0.078947,0.000000,35,38,abs_spearman >= 0.95
666,classification_augmented,redundancy,pymfe__class_conc.quantiles.3,pymfe__class_conc.iq_range,0.954007,0.026316,0.026316,37,37,abs_spearman >= 0.95
667,classification_augmented,redundancy,pymfe__tree_depth.histogram.9,pymfe__leaves_branch.histogram.9,0.951541,0.078947,0.078947,35,35,abs_spearman >= 0.95
668,classification_augmented,redundancy,pymfe__var_importance.median,pymfe__var_importance.iq_range,0.951537,0.078947,0.078947,28,33,abs_spearman >= 0.95


## Estimate Feature-Performance Associations

We now analyze the two reduced matrices separately at the dataset level. Each row must correspond to one independent dataset; split-level rows or repeated datasets would inflate significance and therefore fail the guard below. For each retained meta-feature with at least 30 paired datasets, we estimate its monotonic association with normalized performance difference (`delta_norm`) using Spearman correlation. P-values are adjusted with Benjamini-Hochberg FDR within each analysis table. Dataset-bootstrap confidence intervals and sign-consistency diagnostics are computed for all tested features. For the reported top features, we also record bootstrap rank stability after ranking against the full tested feature set. This keeps the inferential step separate from preprocessing: missingness filtering, near-constant filtering, and redundancy reduction were all target-agnostic.

In [52]:
from scipy import stats
from statsmodels.stats.multitest import multipletests


ASSOCIATION_TARGET = "delta_norm"
ASSOCIATION_MIN_N = 30
ASSOCIATION_ALPHA = 0.05
ASSOCIATION_BOOTSTRAP_REPEATS = 500
ASSOCIATION_CI_TOP_K = None
ASSOCIATION_RANK_STABILITY_TOP_K = 25
ASSOCIATION_RANDOM_SEED = 20260424
INDEPENDENT_UNIT_COLUMN = "dataset"


def assert_dataset_level_table(
    table: pd.DataFrame,
    *,
    table_name: str,
    unit_column: str = INDEPENDENT_UNIT_COLUMN,
) -> dict[str, int | str]:
    if unit_column not in table.columns:
        raise KeyError(
            f"{table_name} is missing the independent unit column: {unit_column}"
        )

    n_rows = len(table)
    n_units = table[unit_column].nunique(dropna=False)
    duplicate_mask = table[unit_column].duplicated(keep=False)
    if duplicate_mask.any():
        duplicate_units = (
            table.loc[duplicate_mask, unit_column].drop_duplicates().head(10).tolist()
        )
        raise ValueError(
            f"{table_name} is not dataset-level: {n_rows} rows but {n_units} unique "
            f"{unit_column} values. Repeated examples: {duplicate_units}. "
            "Aggregate repeated datasets or switch to a cluster-aware bootstrap before association testing."
        )

    return {
        "table": table_name,
        "independent_unit": unit_column,
        "rows": n_rows,
        "unique_units": n_units,
    }


def _association_feature_columns(table: pd.DataFrame) -> list[str]:
    return [
        column
        for column in table.columns
        if column not in available_context_columns and column != ASSOCIATION_TARGET
    ]


def _bootstrap_spearman_summary(
    x_values: np.ndarray,
    y_values: np.ndarray,
    *,
    rng: np.random.Generator,
    n_repeats: int,
    alpha: float,
    observed_r: float,
) -> tuple[float, float, float, int]:
    n = len(x_values)
    bootstrap_r = []
    for _ in range(n_repeats):
        sample_idx = rng.integers(0, n, size=n)
        sample_x = x_values[sample_idx]
        sample_y = y_values[sample_idx]
        if np.unique(sample_x).size < 2 or np.unique(sample_y).size < 2:
            continue
        statistic = stats.spearmanr(sample_x, sample_y).statistic
        if np.isfinite(statistic):
            bootstrap_r.append(float(statistic))

    if len(bootstrap_r) < 20:
        return np.nan, np.nan, np.nan, len(bootstrap_r)

    bootstrap_r = np.asarray(bootstrap_r, dtype=float)
    ci_low, ci_high = np.quantile(bootstrap_r, [alpha / 2, 1 - alpha / 2])
    observed_sign = np.sign(observed_r)
    sign_consistency = (
        np.mean(np.sign(bootstrap_r) == observed_sign) if observed_sign != 0 else np.nan
    )
    return float(ci_low), float(ci_high), float(sign_consistency), len(bootstrap_r)


def _bootstrap_rank_stability(
    table: pd.DataFrame,
    associations: pd.DataFrame,
    *,
    target: str,
    min_n: int,
    n_repeats: int,
    top_k: int,
    random_seed: int,
) -> pd.DataFrame:
    tested = associations.loc[
        associations["spearman_r"].notna(), ["feature", "spearman_r"]
    ]
    if tested.empty:
        return pd.DataFrame(
            columns=[
                "feature",
                "bootstrap_rank_median",
                "bootstrap_rank_q05",
                "bootstrap_rank_q95",
                "bootstrap_top_k_frequency",
            ]
        )

    tracked_features = tested.head(top_k)["feature"].tolist()
    rank_records = {feature: [] for feature in tracked_features}
    top_k_hits = {feature: 0 for feature in tracked_features}
    feature_values = {
        feature: pd.to_numeric(table[feature], errors="coerce").to_numpy(dtype=float)
        for feature in tested["feature"]
    }
    target_values = pd.to_numeric(table[target], errors="coerce").to_numpy(dtype=float)
    rng = np.random.default_rng(random_seed)

    for _ in range(n_repeats):
        sample_idx = rng.integers(0, len(table), size=len(table))
        replicate_rows = []
        sample_y_all = target_values[sample_idx]
        for feature, values in feature_values.items():
            sample_x_all = values[sample_idx]
            valid = np.isfinite(sample_x_all) & np.isfinite(sample_y_all)
            if valid.sum() < min_n:
                continue
            sample_x = sample_x_all[valid]
            sample_y = sample_y_all[valid]
            if np.unique(sample_x).size < 2 or np.unique(sample_y).size < 2:
                continue
            statistic = stats.spearmanr(sample_x, sample_y).statistic
            if np.isfinite(statistic):
                replicate_rows.append((feature, abs(float(statistic))))

        if not replicate_rows:
            continue

        replicate_ranks = pd.Series(
            {feature: abs_r for feature, abs_r in replicate_rows}
        ).rank(method="first", ascending=False)
        replicate_top = set(replicate_ranks[replicate_ranks <= top_k].index)
        for feature in tracked_features:
            if feature in replicate_ranks.index:
                rank_records[feature].append(float(replicate_ranks[feature]))
                top_k_hits[feature] += int(feature in replicate_top)

    rows = []
    for feature in tracked_features:
        ranks = np.asarray(rank_records[feature], dtype=float)
        if len(ranks) < 20:
            rank_median = np.nan
            rank_q05 = np.nan
            rank_q95 = np.nan
            top_k_frequency = np.nan
        else:
            rank_q05, rank_median, rank_q95 = np.quantile(ranks, [0.05, 0.5, 0.95])
            top_k_frequency = top_k_hits[feature] / len(ranks)
        rows.append(
            {
                "feature": feature,
                "bootstrap_rank_median": rank_median,
                "bootstrap_rank_q05": rank_q05,
                "bootstrap_rank_q95": rank_q95,
                "bootstrap_top_k_frequency": top_k_frequency,
            }
        )
    return pd.DataFrame(rows)


def estimate_feature_associations(
    table: pd.DataFrame,
    *,
    table_name: str,
    target: str = ASSOCIATION_TARGET,
    min_n: int = ASSOCIATION_MIN_N,
    bootstrap_repeats: int = ASSOCIATION_BOOTSTRAP_REPEATS,
    ci_top_k: int | None = ASSOCIATION_CI_TOP_K,
    rank_stability_top_k: int = ASSOCIATION_RANK_STABILITY_TOP_K,
    alpha: float = ASSOCIATION_ALPHA,
    random_seed: int = ASSOCIATION_RANDOM_SEED,
) -> pd.DataFrame:
    assert_dataset_level_table(table, table_name=table_name)
    feature_columns = _association_feature_columns(table)
    target_values = pd.to_numeric(table[target], errors="coerce")
    rows = []

    for feature in feature_columns:
        feature_values = pd.to_numeric(table[feature], errors="coerce")
        valid = feature_values.notna() & target_values.notna()
        x = feature_values.loc[valid].to_numpy(dtype=float)
        y = target_values.loc[valid].to_numpy(dtype=float)
        n = len(x)
        feature_n_unique = int(pd.Series(x).nunique(dropna=True)) if n else 0
        target_n_unique = int(pd.Series(y).nunique(dropna=True)) if n else 0

        if n < min_n:
            spearman_r = np.nan
            p_value = np.nan
            reason = f"n < {min_n}"
        elif feature_n_unique < 2:
            spearman_r = np.nan
            p_value = np.nan
            reason = "feature has <2 observed values"
        elif target_n_unique < 2:
            spearman_r = np.nan
            p_value = np.nan
            reason = "target has <2 observed values"
        else:
            result = stats.spearmanr(x, y)
            spearman_r = float(result.statistic)
            p_value = float(result.pvalue)
            reason = "tested"

        rows.append(
            {
                "table": table_name,
                "feature": feature,
                "n": n,
                "feature_n_unique": feature_n_unique,
                "spearman_r": spearman_r,
                "abs_spearman_r": (
                    abs(spearman_r) if np.isfinite(spearman_r) else np.nan
                ),
                "p_value": p_value,
                "p_value_bh": np.nan,
                "bh_reject_0_05": False,
                "ci_low": np.nan,
                "ci_high": np.nan,
                "bootstrap_sign_consistency": np.nan,
                "bootstrap_repeats_used": 0,
                "reason": reason,
            }
        )

    associations = pd.DataFrame(rows)
    valid_p = associations["p_value"].notna()
    if valid_p.any():
        reject, adjusted_p, _, _ = multipletests(
            associations.loc[valid_p, "p_value"],
            alpha=alpha,
            method="fdr_bh",
        )
        associations.loc[valid_p, "p_value_bh"] = adjusted_p
        associations.loc[valid_p, "bh_reject_0_05"] = reject

    associations = associations.sort_values(
        ["abs_spearman_r", "feature"],
        ascending=[False, True],
        na_position="last",
    ).reset_index(drop=True)
    associations["effect_rank"] = np.arange(1, len(associations) + 1)

    ci_features = associations.loc[associations["spearman_r"].notna(), "feature"]
    if ci_top_k is not None:
        ci_features = ci_features.head(ci_top_k)
    for rank_idx, feature in enumerate(ci_features):
        feature_values = pd.to_numeric(table[feature], errors="coerce")
        valid = feature_values.notna() & target_values.notna()
        x = feature_values.loc[valid].to_numpy(dtype=float)
        y = target_values.loc[valid].to_numpy(dtype=float)
        rng = np.random.default_rng(random_seed + rank_idx)
        observed_r = float(
            associations.loc[associations["feature"] == feature, "spearman_r"].iloc[0]
        )
        ci_low, ci_high, sign_consistency, repeats_used = _bootstrap_spearman_summary(
            x,
            y,
            rng=rng,
            n_repeats=bootstrap_repeats,
            alpha=alpha,
            observed_r=observed_r,
        )
        row_idx = associations.index[associations["feature"] == feature][0]
        associations.loc[row_idx, "ci_low"] = ci_low
        associations.loc[row_idx, "ci_high"] = ci_high
        associations.loc[row_idx, "bootstrap_sign_consistency"] = sign_consistency
        associations.loc[row_idx, "bootstrap_repeats_used"] = repeats_used

    rank_stability = _bootstrap_rank_stability(
        table,
        associations,
        target=target,
        min_n=min_n,
        n_repeats=bootstrap_repeats,
        top_k=rank_stability_top_k,
        random_seed=random_seed + 100_000,
    )
    associations = associations.merge(rank_stability, on="feature", how="left")

    print(
        f"{table_name}: tested {valid_p.sum()} predictors; "
        f"{associations['bh_reject_0_05'].sum()} pass BH FDR at alpha={alpha}"
    )
    return associations


association_unit_summary = pd.DataFrame(
    [
        assert_dataset_level_table(
            analysis_general_reduced,
            table_name="general_regression_and_classification",
        ),
        assert_dataset_level_table(
            analysis_classification_reduced,
            table_name="classification_augmented",
        ),
    ]
)

general_associations = estimate_feature_associations(
    analysis_general_reduced,
    table_name="general_regression_and_classification",
)
classification_associations = estimate_feature_associations(
    analysis_classification_reduced,
    table_name="classification_augmented",
)

association_results = pd.concat(
    [general_associations, classification_associations],
    ignore_index=True,
)
association_summary = pd.DataFrame(
    [
        {
            "table": "general_regression_and_classification",
            "predictors_tested": general_associations["p_value"].notna().sum(),
            "bh_reject_0_05": general_associations["bh_reject_0_05"].sum(),
            "max_abs_spearman_r": general_associations["abs_spearman_r"].max(),
            "median_bootstrap_sign_consistency": general_associations[
                "bootstrap_sign_consistency"
            ].median(),
            "median_top_25_frequency": general_associations[
                "bootstrap_top_k_frequency"
            ].median(),
        },
        {
            "table": "classification_augmented",
            "predictors_tested": classification_associations["p_value"].notna().sum(),
            "bh_reject_0_05": classification_associations["bh_reject_0_05"].sum(),
            "max_abs_spearman_r": classification_associations["abs_spearman_r"].max(),
            "median_bootstrap_sign_consistency": classification_associations[
                "bootstrap_sign_consistency"
            ].median(),
            "median_top_25_frequency": classification_associations[
                "bootstrap_top_k_frequency"
            ].median(),
        },
    ]
)

display(association_unit_summary)
display(association_summary)
display(general_associations.head(25))
display(classification_associations.head(25))

general_regression_and_classification: tested 284 predictors; 1 pass BH FDR at alpha=0.05
classification_augmented: tested 556 predictors; 0 pass BH FDR at alpha=0.05


,table,independent_unit,rows,unique_units
0,general_regression_and_classification,dataset,51,51
1,classification_augmented,dataset,38,38


,table,predictors_tested,bh_reject_0_05,max_abs_spearman_r,median_bootstrap_sign_consistency,median_top_25_frequency
0,general_regression_and_classification,284,1,0.510204,0.823,0.452000
1,classification_augmented,556,0,0.558302,0.802,0.306613


,table,feature,n,feature_n_unique,spearman_r,abs_spearman_r,p_value,p_value_bh,bh_reject_0_05,ci_low,ci_high,bootstrap_sign_consistency,bootstrap_repeats_used,reason,effect_rank,bootstrap_rank_median,bootstrap_rank_q05,bootstrap_rank_q95,bootstrap_top_k_frequency
0,general_regression_and_classification,pymfe__attr_ent.skewness,50,50,-0.510204,0.510204,0.000154,0.043639,True,-0.709497,-0.230523,1.000,500,tested,1,4.0,1.00,40.05,0.908
1,general_regression_and_classification,pymfe__cor.histogram.6,42,34,0.458193,0.458193,0.002278,0.215619,False,0.229967,0.629634,1.000,500,tested,2,10.0,1.00,64.00,0.790
2,general_regression_and_classification,pymfe__attr_conc.quantiles.1,50,50,0.428860,0.428860,0.001887,0.215619,False,0.159531,0.691948,1.000,500,tested,3,14.0,1.00,111.00,0.658
3,general_regression_and_classification,pymfe__sparsity.skewness,44,44,0.421424,0.421424,0.004386,0.249149,False,0.144860,0.666552,0.994,500,tested,4,15.0,1.00,131.20,0.628
4,general_regression_and_classification,pymfe__attr_ent.histogram.9,50,47,0.404514,0.404514,0.003572,0.249149,False,0.137640,0.643536,1.000,500,tested,5,19.0,3.00,89.00,0.598
5,general_regression_and_classification,pymfe__sparsity.histogram.0,45,41,0.395309,0.395309,0.007196,0.292873,False,0.130034,0.619581,0.996,500,tested,6,22.0,2.00,154.10,0.540
6,general_regression_and_classification,pymfe__cor.histogram.8,42,30,0.392757,0.392757,0.010086,0.292873,False,0.124359,0.608455,0.998,500,tested,7,24.0,2.00,125.20,0.524
7,general_regression_and_classification,pymfe__kurtosis.histogram.4,41,33,0.385052,0.385052,0.012920,0.292873,False,0.083667,0.610709,0.984,500,tested,8,26.0,2.00,173.05,0.498
8,general_regression_and_classification,pymfe__min.histogram.6,45,11,0.381082,0.381082,0.009803,0.292873,False,0.121375,0.555753,1.000,500,tested,9,25.0,3.00,118.10,0.504
9,general_regression_and_classification,pymfe__cor.histogram.7,42,33,0.375666,0.375666,0.014220,0.292873,False,0.095156,0.580248,0.998,500,tested,10,26.5,3.00,136.05,0.482


,table,feature,n,feature_n_unique,spearman_r,abs_spearman_r,p_value,p_value_bh,bh_reject_0_05,ci_low,ci_high,bootstrap_sign_consistency,bootstrap_repeats_used,reason,effect_rank,bootstrap_rank_median,bootstrap_rank_q05,bootstrap_rank_q95,bootstrap_top_k_frequency
0,classification_augmented,pymfe__tree_shape.histogram.6,35,34,0.558302,0.558302,0.000492,0.165171,False,0.290071,0.759551,1.000,500,tested,1,11.0,1.00,86.10,0.721443
1,classification_augmented,pymfe__nodes_per_level.histogram.2,35,35,0.551541,0.551541,0.000594,0.165171,False,0.267693,0.727997,1.000,500,tested,2,11.0,1.00,104.20,0.725451
2,classification_augmented,pymfe__attr_ent.skewness,37,37,-0.515647,0.515647,0.001089,0.201911,False,-0.740713,-0.204207,0.998,500,tested,3,21.0,1.00,152.00,0.572000
3,classification_augmented,pymfe__naive_bayes.iq_range,34,34,0.505882,0.505882,0.002271,0.315704,False,0.178757,0.743062,0.996,500,tested,4,22.0,1.00,215.60,0.541582
4,classification_augmented,pymfe__cor.histogram.8,32,24,0.490394,0.490394,0.004380,0.341747,False,0.200001,0.700966,1.000,500,tested,5,28.0,1.00,182.00,0.464368
5,classification_augmented,pymfe__sparsity.skewness,33,33,0.473262,0.473262,0.005408,0.341747,False,0.136228,0.713640,0.996,500,tested,6,36.0,1.00,271.60,0.426439
6,classification_augmented,pymfe__nodes_per_level.max,35,35,-0.464986,0.464986,0.004887,0.341747,False,-0.724423,-0.117768,0.998,500,tested,7,33.0,4.00,254.80,0.432866
7,classification_augmented,pymfe__kurtosis.histogram.4,31,26,0.462876,0.462876,0.008739,0.341747,False,0.159310,0.690448,0.992,500,tested,8,39.0,3.00,247.45,0.388889
8,classification_augmented,pymfe__leaves,35,35,-0.454902,0.454902,0.006040,0.341747,False,-0.721249,-0.060953,0.988,500,tested,9,37.0,3.00,288.10,0.408818
9,classification_augmented,pymfe__nodes_per_level.iq_range,35,35,-0.445378,0.445378,0.007337,0.341747,False,-0.701635,-0.099662,0.994,500,tested,10,42.0,6.00,288.10,0.358717


## Select Robust Univariate Associations

We use the association analysis as a reporting step before interpretation or multivariable adjustment. A feature is reported when it passes Benjamini-Hochberg FDR (`q < 0.05`) and has stable bootstrap direction (`bootstrap_sign_consistency >= 0.90`). Reported features are ranked by absolute Spearman association. Bootstrap confidence intervals and rank-stability diagnostics are reported as robustness diagnostics, not as additional hypothesis tests. The regression+classification and classification-augmented matrices are reported separately.

In [ ]:
ROBUST_ASSOCIATION_Q_THRESHOLD = 0.05
ROBUST_ASSOCIATION_MIN_SIGN_CONSISTENCY = 0.90
ROBUST_ASSOCIATION_TOP_N = 25


ROBUST_ASSOCIATION_OUTPUT_COLUMNS = [
    "feature",
    "n",
    "spearman_r",
    "abs_spearman_r",
    "p_value_bh",
    "ci_low",
    "ci_high",
    "bootstrap_sign_consistency",
    "bootstrap_rank_median",
    "bootstrap_rank_q05",
    "bootstrap_rank_q95",
    "bootstrap_top_k_frequency",
]

ROBUST_ASSOCIATION_COLUMN_NAMES = {
    "feature": "feature",
    "n": "n",
    "spearman_r": "spearman_rho",
    "abs_spearman_r": "abs_spearman_rho",
    "p_value_bh": "bh_q_value",
    "ci_low": "bootstrap_ci_low",
    "ci_high": "bootstrap_ci_high",
    "bootstrap_sign_consistency": "bootstrap_sign_consistency",
    "bootstrap_rank_median": "bootstrap_rank_median",
    "bootstrap_rank_q05": "bootstrap_rank_q05",
    "bootstrap_rank_q95": "bootstrap_rank_q95",
    "bootstrap_top_k_frequency": "bootstrap_top_25_frequency",
}


def build_robust_association_table(
    associations: pd.DataFrame,
    *,
    table_name: str,
    q_threshold: float = ROBUST_ASSOCIATION_Q_THRESHOLD,
    min_sign_consistency: float = ROBUST_ASSOCIATION_MIN_SIGN_CONSISTENCY,
    top_n: int = ROBUST_ASSOCIATION_TOP_N,
) -> pd.DataFrame:
    required_columns = {
        "feature",
        "n",
        "spearman_r",
        "abs_spearman_r",
        "p_value",
        "p_value_bh",
        "ci_low",
        "ci_high",
        "bootstrap_sign_consistency",
        "bootstrap_rank_median",
        "bootstrap_rank_q05",
        "bootstrap_rank_q95",
        "bootstrap_top_k_frequency",
    }
    missing_columns = sorted(required_columns.difference(associations.columns))
    if missing_columns:
        raise KeyError(
            f"{table_name} associations are missing required columns: {missing_columns}"
        )

    tested = associations.loc[associations["p_value"].notna()].copy()
    robust = tested.loc[
        tested["p_value_bh"].lt(q_threshold)
        & tested["bootstrap_sign_consistency"].ge(min_sign_consistency)
    ].copy()

    robust = robust.sort_values(
        ["abs_spearman_r", "bootstrap_sign_consistency", "feature"],
        ascending=[False, False, True],
    )

    output_columns = [
        column
        for column in ROBUST_ASSOCIATION_OUTPUT_COLUMNS
        if column in robust.columns
    ]
    table = (
        robust.loc[:, output_columns]
        .head(top_n)
        .rename(columns=ROBUST_ASSOCIATION_COLUMN_NAMES)
    )
    table.insert(0, "analysis_table", table_name)
    return table.reset_index(drop=True)


robust_general_associations = build_robust_association_table(
    general_associations,
    table_name="regression_classification",
)
robust_classification_associations = build_robust_association_table(
    classification_associations,
    table_name="classification_augmented",
)

robust_association_summary = pd.DataFrame(
    [
        {
            "analysis_table": "regression_classification",
            "tested_predictors": general_associations["p_value"].notna().sum(),
            "reported_features": len(robust_general_associations),
            "q_threshold": ROBUST_ASSOCIATION_Q_THRESHOLD,
            "min_bootstrap_sign_consistency": ROBUST_ASSOCIATION_MIN_SIGN_CONSISTENCY,
        },
        {
            "analysis_table": "classification_augmented",
            "tested_predictors": classification_associations["p_value"].notna().sum(),
            "reported_features": len(robust_classification_associations),
            "q_threshold": ROBUST_ASSOCIATION_Q_THRESHOLD,
            "min_bootstrap_sign_consistency": ROBUST_ASSOCIATION_MIN_SIGN_CONSISTENCY,
        },
    ]
)

robust_association_tables = {
    "regression_classification": robust_general_associations,
    "classification_augmented": robust_classification_associations,
}

robust_associations = pd.concat(
    robust_association_tables.values(),
    ignore_index=True,
)


display(robust_association_summary)
display(robust_general_associations)
display(robust_classification_associations)

,analysis_table,tested_predictors,reported_features,q_threshold,min_bootstrap_sign_consistency
0,regression_classification,284,1,0.05,0.9
1,classification_augmented,556,0,0.05,0.9


,analysis_table,feature,n,spearman_rho,abs_spearman_rho,bh_q_value,bootstrap_ci_low,bootstrap_ci_high,bootstrap_sign_consistency,bootstrap_rank_median,bootstrap_rank_q05,bootstrap_rank_q95,bootstrap_top_25_frequency
0,regression_classification,pymfe__attr_ent.skewness,50,-0.510204,0.510204,0.043639,-0.709497,-0.230523,1.0,4.0,1.0,40.05,0.908


,analysis_table,feature,n,spearman_rho,abs_spearman_rho,bh_q_value,bootstrap_ci_low,bootstrap_ci_high,bootstrap_sign_consistency,bootstrap_rank_median,bootstrap_rank_q05,bootstrap_rank_q95,bootstrap_top_25_frequency
